# Aventix Scarlett Real-Data KGRAG
This notebook uses only the real CSV point-cloud dataset. It never edits source data. Per-well estimates are derived from empty-plate subtraction; gravimetric values are plate-level references only.

## 1. Setup & configuration

In [1]:
from pathlib import Path
import os,re,json,hashlib,datetime,warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy import stats
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError: pass

ROOT = Path.cwd()
possible_roots = [
    ROOT / 'dataset/Point_cloud_dataset',
    ROOT.parent / 'dataset/Point_cloud_dataset'
]

DATASET_DIR = None
for candidate in possible_roots:
    if (candidate / 'manifest.csv').exists():
        DATASET_DIR = candidate
        break
    nested = list(candidate.glob('*/manifest.csv')) if candidate.is_dir() else []
    if nested:
        DATASET_DIR = nested[0].parent
        break

if DATASET_DIR is None:
    DATASET_DIR = ROOT / 'dataset/Point_cloud_dataset'

MANIFEST = DATASET_DIR / 'manifest.csv'
WELLS = DATASET_DIR / 'wells.csv'
SCANS = DATASET_DIR / 'scans'
EMPTY = DATASET_DIR / 'empty_reads'
CONFIG = {'TOLERANCE_PCT': 15.0, 'OUTLIER_SIGMA': 3.0, 'MIN_POINTS': 10, 'RADII_MM': {'AB0796': 4.3, 'EP384': 2.0}, 'CALIBRATION_ID': 'SCARLETT-REAL-UNVERIFIED'}
assert MANIFEST.exists() and WELLS.exists(), f'Missing real dataset under {DATASET_DIR}'
print(f'Dataset path successfully resolved: {DATASET_DIR}')


Dataset path successfully resolved: c:\Users\umair\projects\Aventix_soln\dataset\Point_cloud_dataset


## 2. Dataset audit & CSV ingestion

In [2]:
manifest=pd.read_csv(MANIFEST); 
wells=pd.read_csv(WELLS)
print('Dataset:',DATASET_DIR); print('scans:',len(manifest),'well-center rows:',len(wells))
print(manifest[['file','plate','plate_type','liquid','fill_step','empty_read']].to_string(index=False))
assert len(manifest)==15

def load_points(relative_file, folder):
    path=DATASET_DIR/folder/relative_file if not str(relative_file).startswith(str(folder)) else DATASET_DIR/relative_file
    if not path.exists(): path=DATASET_DIR/relative_file
    data=pd.read_csv(path); required={'well','row','col','x_mm','y_mm','z_mm','sensor'}
    missing=required-data.columns
    if missing: raise ValueError(f'{path}: missing {missing}')
    return data


Dataset: c:\Users\umair\projects\Aventix_soln\dataset\Point_cloud_dataset
scans: 15 well-center rows: 2304
                                     file    plate                                   plate_type liquid  fill_step                       empty_read
   scans/W01_EP384_plateA_water_8.9uL.csv  EP384-A             Eppendorf twin.tec PCR plate 384  water          1  empty_reads/E1_EP384_plateA.csv
  scans/W02_EP384_plateA_water_24.7uL.csv  EP384-A             Eppendorf twin.tec PCR plate 384  water          2  empty_reads/E1_EP384_plateA.csv
  scans/W03_EP384_plateA_water_40.1uL.csv  EP384-A             Eppendorf twin.tec PCR plate 384  water          3  empty_reads/E1_EP384_plateA.csv
 scans/W04_AB0796_plateB_water_42.6uL.csv AB0796-B Thermo Abgene AB-0796 (96-well PCR, skirted)  water          1 empty_reads/E2_AB0796_plateB.csv
 scans/W05_AB0796_plateB_water_52.3uL.csv AB0796-B Thermo Abgene AB-0796 (96-well PCR, skirted)  water          2 empty_reads/E2_AB0796_plateB.csv
scans/W06_A

## 3. Point-cloud preprocessing and dual-sensor fusion

In [3]:
def isolate(points, centre, radius):
    cx,cy=centre
    return points[((points.x_mm-cx)**2+(points.y_mm-cy)**2)<=radius**2].copy()

def clean_z(points):
    if len(points)<3: return points.copy()
    z=points.z_mm.astype(float); lo=z.mean()-CONFIG['OUTLIER_SIGMA']*z.std(); hi=z.mean()+CONFIG['OUTLIER_SIGMA']*z.std()
    return points[(z>=lo)&(z<=hi)].copy()

def well_points(scan, well, radius):
    centre=wells[(wells.scan==scan)&(wells.well==well)]
    if centre.empty: return pd.DataFrame(),None
    pts=isolate(load_points(manifest.loc[manifest.file.str.removesuffix('.csv')==scan,'file'].iloc[0],'').rename(columns={'well':'well_label'}), (centre.iloc[0].centre_x_mm,centre.iloc[0].centre_y_mm), radius)
    # CSV well labels are authoritative; geometry isolation protects against boundary bleed.
    return clean_z(pts),centre.iloc[0]


## 4. Empty-plate subtraction and volume estimation

In [4]:
SURFACE_TENSION = {'water': 0.0728, 'dmso': 0.0436}
DENSITY_MAP = {'water': 0.998, 'dmso': 1.100}

def calculate_meniscus_effect(liquid, density_g_per_mL, radius_mm):
    gamma = SURFACE_TENSION.get(str(liquid).lower(), 0.070)
    rho = float(density_g_per_mL) * 1000.0 if pd.notna(density_g_per_mL) else DENSITY_MAP.get(str(liquid).lower(), 1.0) * 1000.0
    g = 9.81
    r_m = radius_mm * 1e-3
    h_m_mm = (2.0 * gamma / (rho * g * r_m)) * 1e3
    h_m_mm = min(h_m_mm, radius_mm * 0.75)
    v_meniscus_uL = 0.5 * np.pi * (radius_mm ** 2) * h_m_mm
    return h_m_mm, v_meniscus_uL

def estimate_well_volume(fluid_height_mm, plate_type, liquid, density_g_per_mL):
    if not np.isfinite(fluid_height_mm) or fluid_height_mm <= 0:
        return np.nan
    if "384" in str(plate_type):
        r0, k = 0.85, 0.11
    else:
        r0, k = 1.65, 0.21
    r_surface = r0 + k * fluid_height_mm
    h_meniscus, v_meniscus = calculate_meniscus_effect(liquid, density_g_per_mL, r_surface)
    v_body = (np.pi / (3.0 * k)) * ((r0 + k * fluid_height_mm) ** 3 - r0 ** 3)
    total_volume = v_body + 0.3 * v_meniscus
    return float(total_volume)

def extract_fluid_surface_z(pts, ptype):
    rim_max = 10.0 if ptype == "EP384" else 13.2
    s2_pts = pts[(pts.sensor == 2) & (pts.z_mm > 5.0) & (pts.z_mm < rim_max)]
    if len(s2_pts) >= 10:
        return float(s2_pts.z_mm.median())
    s1_pts = pts[(pts.sensor == 1) & (pts.z_mm > 1.8) & (pts.z_mm < rim_max)]
    if len(s1_pts) >= 5:
        return float(s1_pts.z_mm.median())
    valid = pts[pts.z_mm < rim_max]
    return float(valid.z_mm.median()) if len(valid) else np.nan

def extract_floor_z(pts, ptype):
    floor_pts = pts[(pts.sensor == 1) & (pts.z_mm < 3.0)]
    if len(floor_pts) >= 5:
        return float(floor_pts.z_mm.median())
    valid = pts[pts.z_mm < 4.0]
    return float(valid.z_mm.median()) if len(valid) else (1.25 if ptype == "EP384" else 0.82)

def path_for(rel):
    p = DATASET_DIR / rel
    if p.exists(): return p
    return DATASET_DIR / 'scans' / Path(rel).name if 'scans' in str(rel) else DATASET_DIR / 'empty_reads' / Path(rel).name

def points_for_scan(file_rel):
    return pd.read_csv(DATASET_DIR / file_rel)

empty_cache = {}
for rel in manifest.empty_read.dropna().unique():
    empty_cache[str(rel)] = pd.read_csv(DATASET_DIR / str(rel))

rows = []
for _, meta in manifest.iterrows():
    scan_id = Path(meta.file).stem
    filled = pd.read_csv(DATASET_DIR / meta.file)
    ptype = "EP384" if "384" in str(meta.plate_type) else "AB0796"
    radius = 1.5 if ptype == "EP384" else 3.0
    liquid = str(meta.liquid).lower()
    density = meta.density_g_per_mL
    empty = empty_cache.get(str(meta.empty_read)) if pd.notna(meta.empty_read) else None
    baseline_est = empty is None
    
    scan_wells = wells[wells.scan == scan_id]
    for _, wr in scan_wells.iterrows():
        fp = clean_z(isolate(filled, (wr.centre_x_mm, wr.centre_y_mm), radius))
        z_surf = extract_fluid_surface_z(fp, ptype)
        if empty is not None:
            ep = clean_z(isolate(empty, (wr.centre_x_mm, wr.centre_y_mm), radius))
            z_floor = extract_floor_z(ep, ptype)
        else:
            z_floor = 1.25 if ptype == "EP384" else 0.82
            
        height = z_surf - z_floor if np.isfinite(z_surf) and np.isfinite(z_floor) else np.nan
        volume = estimate_well_volume(height, ptype, liquid, density)
        h_m, v_m = calculate_meniscus_effect(liquid, density, radius) if np.isfinite(height) else (np.nan, np.nan)
        
        sensors = set(fp.sensor.astype(int)) if len(fp) else set()
        status = 'PASS'; reasons = []
        if len(fp) == 0: status = 'FAIL'; reasons.append('MISSING_WELL')
        if baseline_est: status = 'WARNING'; reasons.append('BASELINE_ESTIMATED')
        if len(sensors) < 2 and len(fp) > 0: reasons.append('PARTIAL_SCAN')
        
        well_row = ord(wr.well[0].upper()) - ord('A') + 1
        well_col = int(re.sub(r'[^0-9]', '', wr.well))
        rows.append({
            **meta.to_dict(), 'scan_id': scan_id,
            'well': wr.well, 'row': well_row, 'col': well_col,
            'centre_x_mm': wr.centre_x_mm, 'centre_y_mm': wr.centre_y_mm,
            'radius_mm': radius, 'point_count': len(fp), 'sensor_count': len(sensors),
            'z_floor_mm': z_floor, 'z_surface_mm': z_surf,
            'fluid_height_mm': height, 'meniscus_height_mm': h_m, 'meniscus_volume_uL': v_m,
            'estimated_uL': volume, 'nominal_uL': wr.nominal_uL,
            'baseline_estimated': baseline_est, 'status': status,
            'fault_reason': ';'.join(reasons)
        })

measurements = pd.DataFrame(rows)
measurements['deviation_pct'] = (measurements.estimated_uL - measurements.nominal_uL) / measurements.nominal_uL * 100
measurements['gravimetric_deviation_pct'] = (measurements.estimated_uL - measurements.gravimetric_mean_uL_per_well) / measurements.gravimetric_mean_uL_per_well * 100
measurements.head()
measurements.to_csv("measurements.csv")
# print(measurements.groupby('plate')[['estimated_uL', 'nominal_uL', 'gravimetric_mean_uL_per_well', 'gravimetric_deviation_pct']].mean().round(2))


# The difference in Nominal vs Estimated volumes of the reagent

In [5]:
doubt=measurements[["nominal_uL","estimated_uL"]]
print(measurements["estimated_uL"].shape)
doubt.head()

(2304,)


,nominal_uL,estimated_uL
0,8,9.759393
1,8,9.296752
2,8,9.756617
3,8,9.746593
4,8,9.828386


## 5. Deterministic verification
added for the crosss verification of the liquid dispensed and a deterministic ground truth for the AI to work on.

the for loop is building a sort of rule book for individual wells so that AI could work it out more easily

In [6]:
def verify_group(g):
    g=g.copy(); 
    g['rule_id']=''; 
    g['fault_reason']=''; 
    for i,r in g.iterrows():
        reasons=[]
        if pd.isna(r.estimated_uL): reasons.append(('RULE_9','MISSING_WELL','no fluid points'))
        elif abs(r.gravimetric_deviation_pct)>CONFIG['TOLERANCE_PCT']: reasons.append(('RULE_1','GRAVIMETRIC_DEVIATION','deviation from plate gravimetric mean'))
        if r.point_count<CONFIG['MIN_POINTS']: reasons.append(('RULE_3','BUBBLE_OR_MISREAD','low point count'))
        if r.sensor_count<2 and r.point_count>0: reasons.append(('RULE_4','PARTIAL_SCAN','only one sensor returned points'))
        if r.baseline_estimated: reasons.append(('RULE_6','BASELINE_ESTIMATED','no matching empty read'))
        
        unique_reasons=[]
        seen=set()
        for r_id, r_name, r_desc in reasons:
            if r_name not in seen:
                seen.add(r_name)
                unique_reasons.append((r_id, r_name, r_desc))
                
        if unique_reasons:
            g.loc[i,'fault_reason']='; '.join(x[1] for x in unique_reasons)
            g.loc[i,'rule_id']=';'.join(x[0] for x in unique_reasons)
            g.loc[i,'status']='FAIL' if any(x[0] in {'RULE_1','RULE_9'} for x in unique_reasons) else 'WARNING'
        else:
            g.loc[i,'status']='PASS'
            g.loc[i,'fault_reason']=''
            g.loc[i,'rule_id']=''
    return g
measurements=pd.concat([verify_group(g) for _,g in measurements.groupby('scan_id')],ignore_index=True)


## 6. Knowledge graph construction

In [7]:
G = nx.MultiDiGraph()
G.add_node('Lab:Aventix', type='Lab', name='Aventix Precision Lab')
G.add_node('Instrument:SCARLETT', type='Instrument', instrument_type='optical_scanner', manufacturer='Aventix')
G.add_node('Instrument:Combi', type='Instrument', instrument_type='liquid_dispenser', model='Thermo Multidrop Combi')

# 1. Liquids from measurements
for liquid_name, grp in measurements.groupby('liquid'):
    density = grp['density_g_per_mL'].dropna().iloc[0] if 'density_g_per_mL' in grp else 1.0
    gamma = SURFACE_TENSION.get(str(liquid_name).lower(), 0.070)
    G.add_node(f"Liquid:{liquid_name}", type='Liquid', name=str(liquid_name), density_g_per_mL=float(density), surface_tension_N_m=float(gamma))

# 2. Plates & Empty Read baselines from measurements
for plate_id, grp in measurements.groupby('plate'):
    first = grp.iloc[0]
    fmt = 'EP384' if '384' in str(first['plate_type']) else 'AB0796'
    plate_node = f"Plate:{plate_id}"
    G.add_node(plate_node, type='Plate', plate_id=plate_id, plate_type=fmt, wells_count=int(first['wells']))
    if pd.notna(first['empty_read']):
        empty_id = f"EmptyRead:{Path(str(first['empty_read'])).stem}"
        G.add_node(empty_id, type='EmptyRead', file=str(first['empty_read']))
        G.add_edge(plate_node, empty_id, relation='HAS_BASELINE_SCAN')

# 3. Experimental Runs from measurements
for scan_id, grp in measurements.groupby('scan_id'):
    first = grp.iloc[0]
    run_node = f"Run:{scan_id}"
    plate_node = f"Plate:{first['plate']}"
    liquid_node = f"Liquid:{first['liquid']}"
    
    G.add_node(run_node, type='Run', scan_id=scan_id, file=first['file'], fill_step=int(first['fill_step']),
               layout=str(first.get('layout', '')), scan_start_utc=str(first['scan_start_utc']))
    G.add_edge('Lab:Aventix', run_node, relation='CONDUCTED_RUN')
    G.add_edge(run_node, plate_node, relation='TARGETED_PLATE')
    G.add_edge(run_node, liquid_node, relation='USED_LIQUID')
    G.add_edge(run_node, 'Instrument:Combi', relation='DISPENSED_BY')

# 4. Wells, Measurements, and Dispenser Channels from measurements
for _, r in measurements.iterrows():
    plate_node = f"Plate:{r['plate']}"
    well_node = f"Well:{r['plate']}:{r['well']}"
    meas_node = f"Measurement:{r['plate']}:{r['scan_id']}:{r['well']}"
    fmt = 'EP384' if '384' in str(r['plate_type']) else 'AB0796'
    
    # Well entity
    G.add_node(well_node, type='Well', plate_id=r['plate'], well_id=r['well'],
               row=int(r['row']), col=int(r['col']),
               centre_x_mm=float(r['centre_x_mm']), centre_y_mm=float(r['centre_y_mm']))
    G.add_edge(plate_node, well_node, relation='HAS_WELL')
    
    # Measurement entity
    dev = r['gravimetric_deviation_pct'] if pd.notna(r['gravimetric_deviation_pct']) else r['deviation_pct']
    G.add_node(meas_node, type='Measurement',
               scan_id=r['scan_id'], well_id=r['well'], plate_id=r['plate'], source_file=r['file'],
               estimated_uL=float(r['estimated_uL']) if pd.notna(r['estimated_uL']) else np.nan,
               nominal_uL=float(r['nominal_uL']) if pd.notna(r['nominal_uL']) else np.nan,
               deviation_pct=float(dev) if pd.notna(dev) else 0.0,
               status=str(r['status']), fault_reason=str(r['fault_reason']), rule_id=str(r['rule_id']),
               fluid_height_mm=float(r['fluid_height_mm']) if pd.notna(r['fluid_height_mm']) else np.nan,
               meniscus_height_mm=float(r['meniscus_height_mm']) if pd.notna(r['meniscus_height_mm']) else np.nan,
               point_count=int(r['point_count']), sensor_count=int(r['sensor_count']),
               baseline_estimated=bool(r['baseline_estimated']))
               
    G.add_edge(well_node, meas_node, relation='HAS_MEASUREMENT')
    G.add_edge(meas_node, 'Instrument:SCARLETT', relation='MEASURED_BY')
    
    # Dispenser Channel
    n_channels = 16 if fmt == 'EP384' else 8
    chan_num = (int(r['col']) - 1) % n_channels + 1
    chan_node = f"DispenserChannel:{chan_num}"
    G.add_node(chan_node, type='DispenserChannel', channel_number=chan_num, instrument='Instrument:Combi')
    G.add_edge(meas_node, chan_node, relation='DISPENSED_BY_CHANNEL')

print(f"Knowledge Graph successfully built from measurements: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


Knowledge Graph successfully built from measurements: 3410 nodes, 9282 edges


## 7. Grounded RAG query layer

In [8]:
def retrieve(question):
    q = question.lower()
    
    # 1. Match plates (full name or short name e.g. 'plate B', 'EP384-A')
    plate_hits = set()
    for p in manifest.plate.unique():
        if p.lower() in q:
            plate_hits.add(p)
    short_matches = re.findall(r'\bplate\s*[-_]?\s*([a-h]|ep384[-_]?a?)\b', q)
    for m in short_matches:
        if m.startswith('ep'):
            plate_hits.add('EP384-A')
        else:
            for p in manifest.plate.unique():
                if p.endswith(m.upper()):
                    plate_hits.add(p)
                    
    # 2. Match well IDs (rows A through P for 96 and 384 well plates)
    well_hits = [w.upper() for w in re.findall(r'\b([a-p]\d{1,2})\b', q)]
    
    results = []
    for n, d in G.nodes(data=True):
        if d.get('type') != 'Measurement':
            continue
        if plate_hits and d.get('plate_id') not in plate_hits:
            continue
        if well_hits and d.get('well_id') not in well_hits:
            continue
        if any(term in q for term in ['failed', 'fault', 'anomal', 'issue']) and d.get('status') not in {'FAIL', 'WARNING', 'UNVERIFIED'}:
            continue
        results.append({'measurement_id': n, **d})
        
    return results[:50]

def ask(question, use_llm=True):
    facts = retrieve(question)
    if not facts:
        return {'answer': 'No verified data found in the graph for this query.', 'evidence_pack': []}
    
    if use_llm and os.getenv('GROQ_API_KEY'):
        try:
            from groq import Groq
            context = '\n'.join(
                f"- {f['measurement_id']}: Well {f.get('well_id')} on Plate {f.get('plate_id')} (Scan {f.get('scan_id')}). Status={f.get('status')}, Est={f.get('estimated_uL', np.nan):.2f} uL, Nom={f.get('nominal_uL', np.nan):.2f} uL, Dev={f.get('deviation_pct', 0.0):.1f}%. Fault={f.get('fault_reason', 'none')}."
                for f in facts[:20]
            )
            p = f"Answer in 1-3 concise natural language sentences using only these graph facts. Explain clearly why the well failed or passed, citing Measurement IDs. Do not invent causes or numbers.\nQuestion: {question}\nFacts:\n{context}"
            model_name = os.getenv('MODEL', 'openai/gpt-oss-120b')
            client = Groq(api_key=os.environ['GROQ_API_KEY'])
            a = client.chat.completions.create(
                model=model_name,
                temperature=0.1,
                messages=[
                    {'role': 'system', 'content': 'You are a grounded laboratory QC consultant.'},
                    {'role': 'user', 'content': p}
                ]
            ).choices[0].message.content.strip()
            return {'answer': a, 'evidence_pack': facts}
        except Exception as e:
            print(f"[Notice: LLM call fallback due to: {e}]")
            
    # Deterministic Grounded Synthesis (Fallback)
    lines = []
    lines.append(f"Grounded Findings ({len(facts)} measurement{'s' if len(facts) > 1 else ''} retrieved):")
    for f in facts[:10]:
        status = f.get('status', 'UNKNOWN')
        well = f.get('well_id', 'N/A')
        plate = f.get('plate_id', 'N/A')
        scan = f.get('scan_id', 'N/A')
        est = f.get('estimated_uL', np.nan)
        nom = f.get('nominal_uL', np.nan)
        dev = f.get('deviation_pct', 0.0)
        reasons = f.get('fault_reason', '').strip() or 'None'
        rules = f.get('rule_id', '').strip() or 'None'
        vol_info = f"Measured: {est:.2f} µL (Nominal: {nom:.2f} µL, Dev: {dev:+.1f}%)" if pd.notna(est) else f"Measured: {est}"
        lines.append(f"• Well {well} ({plate}, Scan {scan}): Status {status} | {vol_info} | Rules: [{rules}] | Reasons: {reasons}")
    if len(facts) > 10:
        lines.append(f"... and {len(facts) - 10} more measurements.")
    return {'answer': '\n'.join(lines), 'evidence_pack': facts}


## 8. Visualisation & validation

In [9]:
summary_df = measurements.groupby('plate').agg(
    scans=('scan_id', 'nunique'),
    wells=('well', 'count'),
    mean_uL=('estimated_uL', 'mean'),
    mean_abs_dev_pct=('deviation_pct', lambda x: x.abs().mean())
)
print(summary_df.round(2))
assert set(measurements.status).issubset({'PASS', 'FAIL', 'WARNING', 'UNVERIFIED'})
assert measurements[measurements.baseline_estimated].shape[0] > 0


          scans  wells  mean_uL  mean_abs_dev_pct
plate                                            
AB0796-B      4    384   131.55             25.42
AB0796-C      1     96    87.12             30.66
AB0796-D      1     96    42.25             50.07
AB0796-E      3    288   109.45             30.57
AB0796-F      1     96   102.89               inf
AB0796-G      1     96    74.26             32.49
AB0796-H      1     96    41.13             35.23
EP384-A       3   1152    24.76              9.47


## 9. Final real-data demo

In [10]:
result = ask('Why was well B3 faulty on plate AB0796-B?')
print("=== Grounded Query Result ===")
print(result['answer'])
print()
print("Evidence Count:", len(result['evidence_pack']))
print("Evidence IDs:")
for x in result['evidence_pack']:
    print(f" - {x['measurement_id']}")


=== Grounded Query Result ===
Well B3 on plate AB0796‑B failed because every scan (W04_AB0796_plateB_water_42.6uL, W05_AB0796_plateB_water_52.3uL, W06_AB0796_plateB_water_181.8uL, W07_AB0796_plateB_water_311.1uL) returned a FAIL status with a GRAVIMETRIC_DEVIATION fault and large negative deviations from the nominal volumes (Measurement IDs: AB0796‑B:W04_AB0796_plateB_water_42.6uL:B3, AB0796‑B:W05_AB0796_plateB_water_52.3uL:B3, AB0796‑B:W06_AB0796_plateB_water_181.8uL:B3, AB0796‑B:W07_AB0796_plateB_water_311.1uL:B3).

Evidence Count: 4
Evidence IDs:
 - Measurement:AB0796-B:W04_AB0796_plateB_water_42.6uL:B3
 - Measurement:AB0796-B:W05_AB0796_plateB_water_52.3uL:B3
 - Measurement:AB0796-B:W06_AB0796_plateB_water_181.8uL:B3
 - Measurement:AB0796-B:W07_AB0796_plateB_water_311.1uL:B3
